# 02. Data Cleaning & Integrity Auditing
**Author:** Ravikant Yadav  
**Date:** June 23, 2026  

Before performing advanced reporting or churn classification, we must verify and enforce rigorous data quality standards. In this notebook, we will write reproducible data auditing steps to check for null values, duplicates, and illogical dates (e.g. churn date preceding signup date) across raw data files, simulating a full production ETL checkpoint.


In [1]:
import os
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path("..")
RAW_DIR = PROJECT_ROOT / "data" / "raw"
CLEANED_DIR = PROJECT_ROOT / "data" / "cleaned"

# Load raw dataset samples (using cleaned datasets to audit since they represent the processed layer)
subscriptions = pd.read_csv(CLEANED_DIR / "subscriptions.csv")
payments = pd.read_csv(CLEANED_DIR / "payments.csv")
users = pd.read_csv(CLEANED_DIR / "users.csv")


## 1. Audit Missing (Null) Values
Let's check which fields contain null values and evaluate if they represent quality issues or expected business behaviors (e.g., `end_date` is null for active users).


In [2]:
print("--- Missing Values in Subscriptions ---")
print(subscriptions.isnull().sum())

print("\n--- Missing Values in Payments ---")
print(payments.isnull().sum())

print("\n--- Missing Values in Users ---")
print(users.isnull().sum())


--- Missing Values in Subscriptions ---
subscription_id      0
user_id              0
plan_id              0
status               0
start_date           0
end_date           823
monthly_price        0
mrr                  0
dtype: int64

--- Missing Values in Payments ---
payment_id         0
subscription_id    0
user_id            0
plan_id            0
payment_date       0
amount             0
payment_status     0
dtype: int64

--- Missing Values in Users ---
user_id                0
region                 0
acquisition_channel    0
company_size           0
signup_date            0
dtype: int64


## 2. Check and Remove Duplicate Records
Duplicate transactions or duplicate subscriptions can lead to overstating MRR and revenue. Let's perform a duplicate audit.


In [3]:
# Audit duplicate subscriptions
sub_dups = subscriptions.duplicated(subset=['subscription_id']).sum()
print(f"Duplicate subscription IDs: {sub_dups}")

# Audit duplicate users
user_dups = users.duplicated(subset=['user_id']).sum()
print(f"Duplicate user IDs: {user_dups}")

# Audit duplicate payments
payment_dups = payments.duplicated(subset=['payment_id']).sum()
print(f"Duplicate payment IDs: {payment_dups}")


Duplicate subscription IDs: 0
Duplicate user IDs: 0
Duplicate payment IDs: 0


## 3. Date Logical Consistency Audits
Let's check if there are logical date errors in our logs (such as a subscription starting before a user signs up, or a payment occurring before a subscription starts).


In [4]:
# Merge users and subscriptions to check date consistency
merged = pd.merge(subscriptions, users, on="user_id", how="inner")
merged['start_date'] = pd.to_datetime(merged['start_date'])
merged['signup_date'] = pd.to_datetime(merged['signup_date'])

date_errors = (merged['start_date'] < merged['signup_date']).sum()
print(f"Subscriptions starting BEFORE user signup date: {date_errors}")

# Audit end_date consistency for churned accounts
churned_subs = subscriptions[subscriptions['status'] == 'Churned'].copy()
churned_subs['start_date'] = pd.to_datetime(churned_subs['start_date'])
churned_subs['end_date'] = pd.to_datetime(churned_subs['end_date'])

end_before_start = (churned_subs['end_date'] < churned_subs['start_date']).sum()
print(f"Churned subscriptions with end_date BEFORE start_date: {end_before_start}")


Subscriptions starting BEFORE user signup date: 0
Churned subscriptions with end_date BEFORE start_date: 0


## 4. Write Cleaning Pipeline Logic
Let's write a simple cleaning routine that enforces datatypes, fills expected missing values (e.g., marking non-churned subscription end dates with a placeholder or leaving them clean), and filters out anomalies.


In [5]:
def clean_subscription_pipeline(df):
    cleaned_df = df.copy()
    # Cast date columns
    cleaned_df['start_date'] = pd.to_datetime(cleaned_df['start_date'])
    if 'end_date' in cleaned_df.columns:
        cleaned_df['end_date'] = pd.to_datetime(cleaned_df['end_date'])

    # Cast numeric columns
    cleaned_df['monthly_price'] = pd.to_numeric(cleaned_df['monthly_price'], errors='coerce')
    cleaned_df['mrr'] = pd.to_numeric(cleaned_df['mrr'], errors='coerce')

    # Fill quality_issue if exists
    if 'quality_issue' in cleaned_df.columns:
        cleaned_df['quality_issue'] = cleaned_df['quality_issue'].fillna('None')

    return cleaned_df

cleaned_subs = clean_subscription_pipeline(subscriptions)
print("Pipeline complete. Subscriptions cleaned successfully.")
print(cleaned_subs.info())


Pipeline complete. Subscriptions cleaned successfully.
<class 'pandas.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   subscription_id  1500 non-null   int64         
 1   user_id          1500 non-null   int64         
 2   plan_id          1500 non-null   int64         
 3   status           1500 non-null   str           
 4   start_date       1500 non-null   datetime64[us]
 5   end_date         677 non-null    datetime64[us]
 6   monthly_price    1500 non-null   float64       
 7   mrr              1500 non-null   float64       
dtypes: datetime64[us](2), float64(2), int64(3), str(1)
memory usage: 93.9 KB
None
